In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("PySpark_JSON_CSV_Handling")\
        .master("local[*]")\
        .getOrCreate()

spark

In [11]:
# 1. Write PySpark code to create a new data frame with the data given below? 

data = [("Spring", 12.3),
("Summer", 10.5),
("Autumn", 8.2),
("Winter", 15.1)]

col = ["season","windspeed"]

df = spark.createDataFrame(data,col)

In [12]:
df.show()

+------+---------+
|season|windspeed|
+------+---------+
|Spring|     12.3|
|Summer|     10.5|
|Autumn|      8.2|
|Winter|     15.1|
+------+---------+



In [15]:
# 2. Consider the library management dataset located at the following path
# (/datasets/library_data.json). Using PySpark, load the
# data into a Dataframe and enforce schema using StructType.

library_df = spark.read.format("json").option("header",True).option("multilinejson",True).load("./data/library_data.json")


In [16]:
library_df.show()

+--------------------+-----------------+-----------+--------------------+
|               books|     library_name|   location|             members|
+--------------------+-----------------+-----------+--------------------+
|[{F. Scott Fitzge...|  Central Library|City Center|[{28, [B001], M00...|
|[{George Orwell, ...|Community Library|     Suburb|[{42, [B003, B004...|
+--------------------+-----------------+-----------+--------------------+



In [17]:
library_df.printSchema()

root
 |-- books: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- author: string (nullable = true)
 |    |    |-- book_id: string (nullable = true)
 |    |    |-- book_name: string (nullable = true)
 |    |    |-- copies_available: long (nullable = true)
 |-- library_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- members: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- age: long (nullable = true)
 |    |    |-- books_borrowed: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- member_id: string (nullable = true)
 |    |    |-- member_name: string (nullable = true)



In [18]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,ArrayType

book_schema = StructType([
    StructField("author",StringType(),True),
    StructField("book_id",StringType(),True),
    StructField("book_name",StringType(),True),
    StructField("copies_available",IntegerType(),True)
 ])

member_schema = StructType([
    StructField("age",IntegerType(),True),
    StructField("book_borrowed",ArrayType(StringType()),True),
    StructField("member_id",StringType(),True),
    StructField("member_name",StringType(),True)
])

In [33]:
_lib_schema = StructType([
    StructField("books",ArrayType(book_schema)),
    StructField("library_name",StringType(),True),
    StructField("location",StringType(),True),
    StructField("members",ArrayType(member_schema))
])

In [34]:
library_df = spark.read.format("json")\
               .option("header",True)\
               .schema(_lib_schema)\
               .option("multilinejson",True)\
               .load("./data/library_data.json")

In [35]:
library_df.printSchema()

root
 |-- books: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- author: string (nullable = true)
 |    |    |-- book_id: string (nullable = true)
 |    |    |-- book_name: string (nullable = true)
 |    |    |-- copies_available: integer (nullable = true)
 |-- library_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- members: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- age: integer (nullable = true)
 |    |    |-- book_borrowed: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- member_id: string (nullable = true)
 |    |    |-- member_name: string (nullable = true)



In [36]:
library_df.show()

+--------------------+-----------------+-----------+--------------------+
|               books|     library_name|   location|             members|
+--------------------+-----------------+-----------+--------------------+
|[{F. Scott Fitzge...|  Central Library|City Center|[{28, null, M001,...|
|[{George Orwell, ...|Community Library|     Suburb|[{42, null, M003,...|
+--------------------+-----------------+-----------+--------------------+



In [40]:
# 3. Given the dataset (/datasets/train.csv), create a
# Data frame using PySpark and perform the following operations
# a) Drop the columns passenger_name and age from the dataset.
# b) Count the number of rows after removing duplicates of columns
# train_number and ticket_number.
# c) Count the number of unique train names.

train_df = spark.read.csv("./data/train.csv",header=True,inferSchema=True)

In [41]:
train_df.show()

+------------+----------+---------------+--------------+---+-------------+-----------+
|train_number|train_name|seats_available|passenger_name|age|ticket_number|seat_number|
+------------+----------+---------------+--------------+---+-------------+-----------+
|         123|   Express|            100|          John| 25|         T123|         A1|
|         123|   Express|            100|          Emma| 30|         T124|         B2|
|         456| Superfast|            150|       Michael| 35|         T125|         C3|
|         456| Superfast|            150|        Sophia| 40|         T126|         D4|
|         789|     Local|             50|       William| 28|         T127|         E5|
|         789|     Local|             50|        Sophia| 32|         T128|         F6|
|         789|     Local|             50|        Oliver| 45|         T129|         G7|
+------------+----------+---------------+--------------+---+-------------+-----------+



In [47]:
droped_df = train_df.dropDuplicates(["passenger_name","age"])

In [48]:
droped_df.count()

7

In [51]:
df = droped_df.distinct()

In [53]:
df.show()
df.count()

+------------+----------+---------------+--------------+---+-------------+-----------+
|train_number|train_name|seats_available|passenger_name|age|ticket_number|seat_number|
+------------+----------+---------------+--------------+---+-------------+-----------+
|         123|   Express|            100|          Emma| 30|         T124|         B2|
|         123|   Express|            100|          John| 25|         T123|         A1|
|         456| Superfast|            150|       Michael| 35|         T125|         C3|
|         789|     Local|             50|        Oliver| 45|         T129|         G7|
|         789|     Local|             50|        Sophia| 32|         T128|         F6|
|         456| Superfast|            150|        Sophia| 40|         T126|         D4|
|         789|     Local|             50|       William| 28|         T127|         E5|
+------------+----------+---------------+--------------+---+-------------+-----------+



7